In [1]:
# --- Celda 1.1: Importaciones y Carga de Librerías Offline ---
import os
import numpy as np
import pandas as pd
import glob
import ROOT
import time
import traceback
from multiprocessing import Pool, cpu_count
from functools import partial

Welcome to JupyROOT 6.30/04


In [2]:
# Esta es la parte MÁS IMPORTANTE:
# Asegúrate de que estás corriendo este Jupyter Lab desde una terminal
# donde ANTES hiciste: source /ruta/a/auger/offline/this-auger-offline.sh

AugerOfflineRoot = os.environ.get("AUGEROFFLINEROOT")
if AugerOfflineRoot is None:
    raise EnvironmentError(
        "AUGEROFFLINEROOT no definido. "
        "Reinicia Jupyter Lab desde una terminal donde hayas "
        "hecho: "
        " 'aug_set_version offline 4.0.1-icrc23-prod1-root6' "   
        " 'source /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6/bin/this-auger-offline.sh'."
    )

print(f"AUGEROFFLINEROOT encontrado en: {AugerOfflineRoot}")

# Cargar las librerías necesarias
print("Cargando librerías de Auger Offline...")
libs_to_load = ["libRecEventKG.so"]
for lib in libs_to_load:
    lib_path = os.path.join(AugerOfflineRoot, "lib", lib)
    if not os.path.exists(lib_path):
        raise FileNotFoundError(f"No se encontró la librería: {lib_path}")
    
    # Usamos gSystem.Load que es más robusto en PyROOT
    status = ROOT.gSystem.Load(lib_path)
    if status < 0:
        raise ImportError(f"Error cargando la librería: {lib_path}")

print("Librerías cargadas correctamente. ¡Listo para trabajar! 🚀")

AUGEROFFLINEROOT encontrado en: /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6
Cargando librerías de Auger Offline...
Librerías cargadas correctamente. ¡Listo para trabajar! 🚀


In [3]:
# --- Celda 2.1: Funciones Auxiliares y Principales (con nMuones_MC) ---

def getModuleList(counter, sim=True):
    """
    Obtiene la lista de objetos 'Module' (segmentos de detector)
    asociados a un 'Counter' (estación UMD).
    
    Parameters:
    ----------
    counter : ROOT.mevt.Counter
        La estación UMD de la cual extraer los módulos.
    sim : bool, default=True
        Flag para indicar si son datos de simulación.
        - True (Simulación): IDs de módulo son 0, 1, 2...
        - False (Datos Reales): IDs de módulo son 100, 101, 102...
    """
    possibleModules = range(0, 6) if sim else range(100, 116)
    modules = []
    for modId in possibleModules:
        if counter.HasModule(modId):
            modules.append(counter.GetModule(modId))
    return modules

def readADST_surface_v7(fname, is_mc_simulation=True):
    """
    Leer un archivo ADST (1 fila por MÓDULO).
    
    Esta función es el corazón del pipeline de procesamiento. Itera sobre
    cada evento (lluvia) en un archivo ADST y extrae la información
    relevante a nivel de MÓDULO de UMD (el nivel más granular).
    
    Lógica Clave:
    1. Itera sobre el MDEvent (mEvent.CountersBegin()) para encontrar
       TODOS los counters UMD (tanto Infill '104k' como Anillo Denso '90k').
    2. Usa el SDEvent como "diccionario" de geometría para obtener r y phi.
    3. NO filtra por 'IsLowGainSaturated', en su lugar, guarda un flag.
    4. Corrige el cálculo de 'phi_plane' para el Anillo Denso (90k).
    5. Guarda la señal REC ('nMuones_REC') y la señal MC ('nMuones_MC')
       para cada módulo individual.
    """
    
    print(f"Iniciando lectura de: {os.path.basename(fname)}")
    
    if not os.path.exists(fname):
        print(f"Advertencia: Archivo no encontrado {fname}")
        return pd.DataFrame() # Retorna DF vacío

    # --- Inicialización de ROOT ---
    files = ROOT.std.vector('string')()
    files.push_back(fname)

    file1 = ROOT.RecEventFile(files)
    event = ROOT.RecEvent()
    geo = ROOT.DetectorGeometry()
    
    file1.ReadDetectorGeometry(geo) # Ignoramos fallos (Warnings de TStreamerInfo)
    file1.SetBuffers(event)

    data = [] 
    event_count = 0
    start_time = time.time()

    # --- COMIENZA EL BUCLE DE EVENTOS ---
    while file1.ReadNextEvent() == ROOT.RecEventFile.eSuccess:
        event_count += 1
        if event_count % 500 == 0:
            print(f"... procesados {event_count} eventos.")

        # --- Info Global del Evento (Lluvia) ---
        event_id_lluvia = event.GetEventId()
        
        MCShower = event.GetGenShower()
        mc_energy = MCShower.GetEnergy()
        logE_MC = np.log10(mc_energy) if mc_energy > 0 else np.nan
        theta_MC = MCShower.GetZenith() * 180.0 / np.pi
        phi_MC = MCShower.GetAzimuth() * 180.0 / np.pi
        primary = MCShower.GetShortPrimaryName()
        
        sEvent = event.GetSDEvent()
        sShower = sEvent.GetSdRecShower()
        rec_energy = sShower.GetEnergy()
        logE_REC = np.log10(rec_energy) if rec_energy > 0 else np.nan
        theta_REC = sShower.GetZenith() * 180.0 / np.pi
        phi_REC = sShower.GetAzimuth() * 180.0 / np.pi
        
        # Azimut de la lluvia REC. Lo necesitamos para el cálculo de phi_rel
        shower_azimuth_rec = sShower.GetAzimuth()
        
        # --- Bucle sobre los COUNTERS (Estaciones UMD) ---
        mEvent = event.GetMDEvent()
        counterIterator = mEvent.CountersBegin()
        countersEnd = mEvent.CountersEnd()
        
        while counterIterator != countersEnd:
            
            counter = counterIterator.__deref__() # Objeto Counter (estación)
            counterId = counter.GetId()           
            sdId = counter.GetSdPartnerId()    

            # Buscamos la estación de superficie (SD) asociada
            sdStation = sEvent.GetStationById(sdId) if sEvent.HasStation(sdId) else None
            
            # --- Corte de Calidad 1: Geometría ---
            # Si no hay sdStation, no podemos calcular r_core ni phi_plane.
            # Ese counter no sirve para el análisis de asimetría.
            if sdStation is None:
                counterIterator += 1 # Avanzamos al siguiente counter
                continue 
            
            # --- Corte de Calidad 2: Saturación (Corte Diferido) ---
            # Guardamos el flag, como discutimos, en lugar de filtrar.
            is_sd_saturated = sdStation.IsLowGainSaturated()

            # --- Info de Geometría y Señal (Nivel Estación) ---
            r = sdStation.GetSPDistance()
            r_core = r
            r_core_err = sdStation.GetSPDistanceError()
            sdSignal = sdStation.GetTotalSignal()
            sdSignal_err = sdStation.GetTotalSignalError()
            sdMuonSignal = sdStation.GetMuonSignal()
            
            # --- ❗️ CORRECCIÓN DE PHI (¡Clave!) ❗️ ---
            station_azimuth_sp = sdStation.GetAzimuthSP()
            
            if sdId >= 90000 and sdId < 100000:
                # ANILLO DENSO (90k): El ángulo ya es relativo al eje de la lluvia.
                phi_rel = station_azimuth_sp
            else:
                # INFILL (104k): El ángulo es absoluto (al Norte), 
                # lo hacemos relativo restando el azimut de la lluvia.
                phi_rel = station_azimuth_sp - shower_azimuth_rec

            # Calculamos X, Y (en plano de lluvia) y phi_plane [0, 2*pi]
            x_plane = r * np.cos(phi_rel)
            y_plane = r * np.sin(phi_rel)
            phi_plane = (phi_rel + 2 * np.pi) % (2 * np.pi)
            
            # --- OBTENER INFO MC (Nivel Estación) ---
            # Necesitamos el "gemelo" de simulación del counter
            simCounter = mEvent.GetSimCounter(counterId)
            if simCounter is None:
                # Si no hay simCounter (raro), no podremos obtener
                # nMuones_MC. Saltamos esta estación.
                counterIterator += 1
                continue
            
            # --- Bucle sobre los MÓDULOS (Segmentos) ---
            modules = getModuleList(counter, sim=is_mc_simulation)
            for module in modules:
                
                # --- Señal Reconstruida (REC) ---
                nMuones_REC = module.GetNumberOfEstimatedMuons()
                moduleId = module.GetId()

                # --- Estado del Módulo (Flag de Calidad) ---
                if module.IsCandidate(): status = "candidate"
                elif module.IsSaturated(): status = "saturated"
                elif module.IsRejected(): status = "rejected"
                elif module.IsSilent(): status = "silent"
                else: status = "undefined"
                
                # --- ❗️ CÁLCULO DE MUONES MC (Verdad) ❗️ ---
                # Inspo Carmi: Iteramos por los 2 canales (scintillators)
                # de este módulo y sumamos los muones MC "inyectados".
                nMuones_MC_module = 0.0
                channelIterator = module.ChannelsBegin()
                channelsEnd = module.ChannelsEnd()
                
                while channelIterator != channelsEnd:
                    channel = channelIterator.__deref__()
                    channelId = channel.GetId()
                    
                    # Le preguntamos al 'simCounter' por el 'simScintillator'
                    # que corresponde a este 'moduleId' y 'channelId'
                    if simCounter.HasSimScintillatorByChannel(moduleId, channelId):
                        mdSimScintillator = simCounter.GetSimScintillatorByChannelId(moduleId, channelId)
                        # ¡Obtenemos la verdad MC!
                        nMuones_MC_module += mdSimScintillator.GetNumberOfInjectedMuons()
                    
                    channelIterator += 1
                # --- ❗️ FIN DEL CÁLCULO MC ❗️ ---


                data.append({
                    "event_id": event_id_lluvia,
                    
                    # Info MC
                    "logE_MC": logE_MC, "theta_MC": theta_MC, "phi_MC": phi_MC, "primary": primary,
                    
                    # Info REC
                    "logE_REC": logE_REC, "theta_REC": theta_REC, "phi_REC": phi_REC,
                    
                    # Info Módulo/Counter
                    "counterId": counterId,
                    "moduleId": moduleId,
                    "nMuones_REC": nMuones_REC,   # <-- ❗️ RENOMBRADO
                    "nMuones_MC": nMuones_MC_module, # <-- ❗️ NUEVA COLUMNA
                    "module_status": status,      
                    "is_sd_saturated": is_sd_saturated, 
                    
                    # Info de Geometría (plano de lluvia)
                    "x_plane": x_plane,
                    "y_plane": y_plane,
                    "phi_plane": phi_plane, # <-- ¡Ahora está bien calculado!
                    "r_core": r_core,
                    "r_core_err": r_core_err,     
                    
                    # Info de Señal (SD)
                    "sdId": sdId,
                    "sdSignal": sdSignal,
                    "sdSignal_err": sdSignal_err,
                    "sdMuonSignal": sdMuonSignal  
                })

                
            counterIterator += 1 # Avanzamos al siguiente counter
        
    end_time = time.time()
    elapsed = end_time - start_time
    
    print(f"Lectura completa. Total de eventos leídos: {event_count}")
    print(f"Tiempo total de lectura: {elapsed:.2f} segundos.")
    print(f"Total de 'MÓDULOS' (filas) extraídos: {len(data)}")

    df = pd.DataFrame(data)
    return df

In [4]:
# -----------------------------------------------------------------
# FUNCIÓN "TRABAJADORA"
# ❗️ 2. AHORA ACEPTA 'output_dir' COMO ARGUMENTO
# -----------------------------------------------------------------
def process_file_wrapper(root_fpath, output_dir):
    
    # 1. Definir rutas
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    # 2. Evitar reprocesar
    if os.path.exists(output_path):
        return f"INFO: El archivo ya existe, saltando: {output_filename}"

    # 3. Imprimir estado
    print(f"► [Iniciando]: {filename}")
    
    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (¡Llama a la función de la Celda 2.1!)
        # (Asegurate de que tu Celda 2.1 se llame 'readADST_surface' 
        # o cambiá el nombre aquí abajo)
        df = readADST_surface_v7(root_fpath) 
        
        if df.empty:
            return f"INFO: Archivo vacío o sin datos UMD. Saltando: {filename}"

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata en {filename}: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        elapsed = end_file_time - start_file_time
        return f"✔ [Éxito]: {filename} -> {output_filename} ({elapsed:.2f}s)"

        # ----- FIN DEL TRABAJO -----

    except Exception as e:
        # Si algo falla, retornamos el string de error
        return f"❌ [ERROR] en {filename}: {e}\n{traceback.format_exc()}"

# Paralelizacion -> QGS-Helio-17.5-18eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_qgs_helium_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")
    
    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")

    # CREAMOS LA FUNCIÓN "PARCIAL" 
    # "Congelamos" el argumento 'output_dir' de nuestra función wrapper
    process_func = partial(process_file_wrapper, output_dir=output_dir)

    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_func, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module_v4/parquet_qgs_helium_17/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root



Iniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfi

# Paralelizacion -> Sibyl-Helio-17.5-18eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_sib_helio_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

# Paralelizacion -> Sibyl-Hierro-17.5-18eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_sib_hierro_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

# Paralelizacion -> Sibyl-Oxigeno-17.5-18eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxigen"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_sib_oxigeno_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

# Paralelizacion -> Sibyl-Proton-17.5-18eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_sib_proton_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

# Paralelizacion -> EPOS-Helio-18-18.5eV

In [ ]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_module_v4/parquet_epos_helio_18/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")